In [1]:
import json
import os
import shutil
from pathlib import Path

# --- Configuration ---
PNG_ROOT = "/weka/kanpur/data_radiovision/paediatric_xray_dataset/physionet.org/png_images/train"
QA_JSONL_PATH = "/nuvodata/User_data/shiva/Grounding-Jepa/Sample_Data/structured_medgemma_results.jsonl"
DEST_FOLDER = "/nuvodata/User_data/shiva/Grounding-Jepa/Sample_Data/images"

# Create destination folder if it doesn't exist
os.makedirs(DEST_FOLDER, exist_ok=True)

# --- Execution ---
print(f"Starting image copy to {DEST_FOLDER}...")

count = 0
not_found = []

with open(QA_JSONL_PATH, 'r') as f:
    for line in f:
        try:
            data = json.loads(line.strip())
            image_id = data.get("image_id")
            
            if image_id:
                # Construct filenames
                # Assuming your PNGs follow the 'image_id.png' naming convention
                src_filename = f"{image_id}.png"
                src_path = os.path.join(PNG_ROOT, src_filename)
                dst_path = os.path.join(DEST_FOLDER, src_filename)
                
                if os.path.exists(src_path):
                    shutil.copy2(src_path, dst_path)
                    count += 1
                else:
                    not_found.append(image_id)
                    
        except json.JSONDecodeError:
            print("Skipping empty or malformed line.")

print(f"\nTask Complete!")
print(f"Successfully copied: {count} images.")

if not_found:
    print(f"Warning: {len(not_found)} images were not found in source directory.")
    # Optional: Print first few missing IDs
    print(f"Missing IDs sample: {not_found[:5]}")

Starting image copy to /nuvodata/User_data/shiva/Grounding-Jepa/Sample_Data/images...

Task Complete!
Successfully copied: 50 images.


In [3]:
import requests
from PIL import Image
from torch.nn.functional import cosine_similarity

from transformers import AutoModel, AutoProcessor

url_1 = "http://images.cocodataset.org/val2017/000000039769.jpg"
url_2 = "http://images.cocodataset.org/val2017/000000219578.jpg"
image_1 = Image.open(requests.get(url_1, stream=True).raw)
image_2 = Image.open(requests.get(url_2, stream=True).raw)

model_id = "jmtzt/ijepa_vitg16_22k"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)


def infer(image):
    inputs = processor(image, return_tensors="pt")
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1)


embed_1 = infer(image_1)
embed_2 = infer(image_2)

similarity = cosine_similarity(embed_1, embed_2)
print(similarity)


preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/513 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


model.safetensors:   0%|          | 0.00/4.05G [00:00<?, ?B/s]

tensor([0.4319], grad_fn=<SumBackward1>)
